# Correlation analysis - Single Video

One row per **detected face**, for **every** frame. Audio is left-joined (NaN where a frame has no clean segment) and flagged with `has_audio`, so visual-only analyses use everything and audio analyses just filter.

In [ ]:
#imports
import pandas as pd
import os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

## Select Candidates

In [ ]:
# Select candidates 

person1 = 'Martins'
person2 = 'Gouveia_Melo'

## Pipeline alternative — optional shortcut using `pipelines.py`

Everything from the cell marked **▼ START** below down to the cell marked **▲ END** (the current cells 6–16, about 230 lines) builds the dataset by hand: it loads the pkl files, extracts features frame by frame, runs the GMM visual classifier, joins audio speaker labels, and aggregates everything to segment level.

Dinis extracted all of that logic into shared functions in `multivideo_analysis/pipelines.py`. If you want to try the centralised version, comment out all cells from **▼ START** to **▲ END** and run the commented code cell just below this text instead — it calls the same logic and produces the same `seg`, `emotion_cols`, and `pose_feature_cols` variables that Sections B, C, and D expect.

**Why Section A is a problem with the pipeline:**
Section A uses `ident`, which comes from `df_all` — a frame-level table (one row per detected face per frame) that also carries audio features and speaker identity. The pipeline function `build_seg_all()` builds that table internally but only *returns* `seg`, the segment-level summary. It throws away the frame-level rows on purpose: when running across all 28 debates keeping every frame in memory would be expensive. For a single-debate analysis like this one that is the wrong tradeoff, so for now Section A still needs the manual cells below. Everything from Section B onwards works perfectly with the pipeline.

In [ ]:
## ── PIPELINE ALTERNATIVE ─────────────────────────────────────────────────────
## Comment out cells 6–16 (from ▼ START to ▲ END), then uncomment this block.
## Requires multivideo_analysis/pipelines.py (run Jupyter from the project root).
##
# import sys, os
# sys.path.insert(0, '../multivideo_analysis')   # one level up from labeling_visual/
# from pipelines import (build_labeled_audio, label_debate_visual,
#                        build_seg_all, NON_REDUNDANT)
#
# # Build the exact video filename from person1 / person2 (defined in cell 4 above)
# video_name = next(
#     f.replace('_audio.pkl', '')
#     for f in os.listdir('Project_Features')
#     if f.endswith('_audio.pkl') and person1 in f and person2 in f
# )
#
# data_audio = build_labeled_audio()
# # ^ loads all 28 debates + runs k=3 speaker embedding clustering
#
# labels = label_debate_visual(video_name, data_audio)
# # ^ visual GMM classifier (landmark ratios -> candidate_left/right/moderator)
# #   then resolves to real names using audio speaker labels
#
# seg = build_seg_all([video_name], data_audio, labels)
# # ^ frame->audio join, identity merge, speaking_face flag, segment aggregation
# #   pass a list of more debate names here for multi-video analysis
#
# # replicate the two column-list variables defined in the manual cells
# emotion_cols      = [c for c in seg.columns if c.startswith('prob_')]
# pose_feature_cols = ['shoulder_slope', 'body_openness', 'head_tilt',
#                      'wrist_height', 'torso_height']
#
# # NOTE: df_all is not available — Section A (ident variable) will not run.
# #       Sections B, C, D work unchanged.

## Load visual + audio for this debate

> **▼ START — manual pipeline** (skip to **▲ END** after the segment-aggregation cell if using the pipeline alternative above)


In [ ]:
# ── Visual ────────────────────────────────────────────────────────────────────
pklfiles_visual = []
for file in os.listdir('Project_Features'):
    if file.endswith('visual.pkl') and person1 in file and person2 in file:
        pklfiles_visual.append(file)

visual_dfs = []
for video in pklfiles_visual:
    print(f"Loading visual: {video}")
    df = pd.read_pickle(os.path.join('Project_Features', video))
    visual_dfs.append(df)

df_visual = pd.concat(visual_dfs, ignore_index=True)
df_visual['video'] = df_visual['Frame'].str.extract(r'Frames/([^/]+)/')
df_visual['frame_number'] = df_visual['Frame'].str.extract(r'frame_(\d+)\.jpg').astype(int)
df_visual = df_visual.sort_values(by=['video', 'frame_number']).reset_index(drop=True)

# ── Audio ─────────────────────────────────────────────────────────────────────
pklfiles_audio = []
for file in os.listdir('Project_Features'):
    if file.endswith('audio.pkl') and person1 in file and person2 in file:
        pklfiles_audio.append(file)

audio_dfs = []
for video in pklfiles_audio:
    print(f"Loading audio: {video}")
    df = pd.read_pickle(os.path.join('Project_Features', video))
    df['video'] = video.replace('_audio.pkl', '')
    audio_dfs.append(df)

df_audio = pd.concat(audio_dfs, ignore_index=True)
df_audio = df_audio.sort_values(by=['video', 'time stamp']).reset_index(drop=True)

def parse_embedding(x):
    if isinstance(x, np.ndarray):
        return x
    return np.fromstring(str(x).strip('[]'), sep=' ')

df_audio['speak_embeddings'] = df_audio['speak_embeddings'].apply(parse_embedding)

## Per-frame audio — keep ALL frames
For each frame second, attach the single audio segment active then. Frames in a gap, or with overlapping segments, keep `has_audio=False` and NaN audio features (we don't drop them).

In [ ]:
NON_REDUNDANT = ['meanF0Hz','stdevF0Hz','HNR','localJitter','localShimmer',
                 'speechrate','npause','f1_mean','f2_mean','fdisp']

df_audio['time_end'] = df_audio['time stamp'] + df_audio['duration']

rows = []
for t in sorted(df_visual['frame_number'].unique()):
    active = df_audio[(df_audio['time stamp'] <= t) & (df_audio['time_end'] > t)]
    if len(active) == 1:                      # exactly one speaker this second
        seg = active.iloc[0]
        r = {'frame_number': int(t), 'has_audio': True, 'segment_id': int(active.index[0])}
        r.update({c: seg[c] for c in NON_REDUNDANT})
    else:                                     # gap (0) or overlap (>1) -> no clean audio
        r = {'frame_number': int(t), 'has_audio': False, 'segment_id': -1}
        r.update({c: np.nan for c in NON_REDUNDANT})
    rows.append(r)

audio_by_frame = pd.DataFrame(rows)
print('frames total:', len(audio_by_frame))
print('frames with audio:', int(audio_by_frame['has_audio'].sum()))
print('distinct segments:', audio_by_frame.loc[audio_by_frame.has_audio, "segment_id"].nunique())

## Per-face visual table (faces matched to bodies + pose features)

In [ ]:
master_data = []
for index, row in df_visual.iterrows():
    frame_id = row['Frame']
    faces = row['Fer']
    poses = row['Poses']
    frame_number = row['frame_number']
    
    if isinstance(faces, list) and len(faces) > 0 and len(faces)<=3:
        valid_faces = [f for f in faces if isinstance(f, dict) and 'bbox' in f]
        people_count = len(valid_faces)
        
        for i, face in enumerate(valid_faces):
            # Face Coordinates
            f_box = face['bbox'] # [x1, y1, x2, y2]
            f_width = f_box[2] - f_box[0]
            f_height = f_box[3] - f_box[1]
            
            f_x_center = f_box[0] + (f_width / 2)
            f_y_center = f_box[1] + (f_height / 2) 
            face_area = f_width * f_height
            
            top_emotion = face.get('top_emotion')
            probability = face['probabilities'].get(top_emotion) if top_emotion and 'probabilities' in face else None
            landmarks = face.get('landmarks')
            
            # MATCH THE BODY TO THE FACE
            matched_body_bbox = None
            matched_pose_keypoints = None
            
            if isinstance(poses, list):
                for person_body in poses:
                    if isinstance(person_body, dict)and len(person_body) > 0:
                        b_box = person_body['bbox'] 
                        
                        # is the center of the face inside this body's bounding box?
                        #note that b_box[3] > bbox[1] because y axis goes down
                        if (b_box[0] <= f_x_center <= b_box[2]) and (b_box[1] <= f_y_center <= b_box[3]):
                            matched_body_bbox = b_box
                            matched_pose_keypoints = person_body.get('pose') # The 17x3 matrix
                            break 
            
            master_data.append({
                'frame': frame_id,
                'frame_number': frame_number,
                'people_count': people_count,
                'person_index': i,
                'face_X_center': f_x_center,
                'face_area': face_area,
                'face_bbox': f_box,
                'body_bbox': matched_body_bbox,
                'top_emotion': top_emotion,
                'landmarks': landmarks,
                'pose_keypoints': matched_pose_keypoints,
                **{f'prob_{e}': p for e, p in face['probabilities'].items()}})
            
    else:
        # Handle empty frames
        master_data.append({
            'frame': frame_id,
            'frame_number': frame_number,
            'people_count': 0,
            'person_index': None,
            'face_X_center': None,
            'face_area': None,
            'face_bbox': None,
            'body_bbox': None,
            'top_emotion': None,
            'landmarks': None,
            'pose_keypoints': None
        })
df_master = pd.DataFrame(master_data)
print(f"Important data (df shape): {df_master.shape}")
print(df_master['pose_keypoints'].isna().sum())
print(f"Total rows: {len(df_master)}")

In [ ]:
KEYPOINTS = {
    'nose': 0, 'left_eye': 1, 'right_eye': 2,
    'left_ear': 3, 'right_ear': 4,
    'left_shoulder': 5, 'right_shoulder': 6,
    'left_elbow': 7, 'right_elbow': 8,
    'left_wrist': 9, 'right_wrist': 10,
    'left_hip': 11, 'right_hip': 12
}

def extract_pose_features(keypoints):
    if keypoints is None:
        return pd.Series({
            'shoulder_slope': None,
            'body_openness': None,
            'head_tilt': None,
            'wrist_height': None,
            'torso_height': None
        })
    
    kp = np.array(keypoints) # 17x3: [x, y, visibility]
    
    l_shoulder = kp[5]
    r_shoulder = kp[6]
    l_wrist = kp[9]
    r_wrist = kp[10]
    l_hip = kp[11]
    r_hip = kp[12]
    nose = kp[0]
    
    # slope between shoulders (positive = left higher, negative = right higher)
    shoulder_slope = r_shoulder[1] - l_shoulder[1]
    
    # horizontal distance between shoulders (wider = more open posture)
    body_openness = abs(r_shoulder[0] - l_shoulder[0])
    
    # nose x relative to shoulder midpoint (head tilt left/right)
    shoulder_mid_x = (l_shoulder[0] + r_shoulder[0]) / 2
    head_tilt = nose[0] - shoulder_mid_x
    
    # average wrist y relative to shoulder y (negative = wrists raised)
    shoulder_mid_y = (l_shoulder[1] + r_shoulder[1]) / 2
    wrist_height = shoulder_mid_y - ((l_wrist[1] + r_wrist[1]) / 2)
    
    # torso height: distance between shoulder midpoint and hip midpoint
    hip_mid_y = (l_hip[1] + r_hip[1]) / 2
    torso_height = abs(hip_mid_y - shoulder_mid_y)
    
    return pd.Series({
        'shoulder_slope': shoulder_slope,
        'body_openness': body_openness,
        'head_tilt': head_tilt,
        'wrist_height': wrist_height,
        'torso_height': torso_height
    })

# Apply to df_master, skipping rows with no pose
df_master_valid = df_master[df_master['pose_keypoints'].notna()].copy()
pose_features = df_master_valid['pose_keypoints'].apply(extract_pose_features)
df_master_valid = pd.concat([df_master_valid, pose_features], axis=1)

print(f"Valid rows for pose-emotion correlation: {len(df_master_valid)}")
print(df_master_valid[['shoulder_slope', 'body_openness', 'head_tilt', 'wrist_height', 'torso_height']].describe().round(2))

## Assemble the master dataset
Per-face visual + identity (`final_name`) + per-frame audio (nullable) + flags. `speaking_face` = this face is the person the audio says is talking.

In [ ]:
emotion_cols      = [c for c in df_master_valid.columns if c.startswith('prob_')]
pose_feature_cols = ['shoulder_slope','body_openness','head_tilt','wrist_height','torso_height']

labels = (pd.read_pickle('labels.pkl')
            .rename(columns={'Frame_Number':'frame_number', 'Person_Index':'person_index'}))
labels = labels.dropna(subset=['person_index']).copy()
labels['frame_number'] = labels['frame_number'].astype(int)
labels['person_index'] = labels['person_index'].astype(int)
df_master_valid['frame_number'] = df_master_valid['frame_number'].astype(int)
df_master_valid['person_index'] = df_master_valid['person_index'].astype(int)

df_all = df_master_valid.merge(
    labels[['frame_number','person_index','final_name','speaker']],
    on=['frame_number','person_index'], how='left')
df_all['speaker'] = df_all['speaker'].replace({'host':'moderator'})

df_all = df_all.merge(audio_by_frame, on='frame_number', how='left')
df_all['has_audio']   = df_all['has_audio'].fillna(False)
df_all['segment_id']  = df_all['segment_id'].fillna(-1).astype(int)
df_all['speaking_face'] = df_all['has_audio'] & (df_all['final_name'] == df_all['speaker'])

print('SHAPE:', df_all.shape, '| identity:', int(df_all.final_name.notna().sum()),
      '| has_audio:', int(df_all.has_audio.sum()), '| speaking:', int(df_all.speaking_face.sum()))

## Segment-level view (for all audio↔visual analysis)
One row per segment: speaking-face emotion/pose **averaged** over the segment, the segment's audio value (`first`, since it's constant), and the segment's dominant emotion.

The audio features are one scalar per segment, but emotion and pose vary frame by frame. To correlate them honestly we need them at the same unit, and that unit has to be the segment (otherwise we'd duplicate each audio value across all its frames). So for every segment we collapse the visual side down to a single summary — the speaking face's average emotion and pose over that segment — and pair it with the segment's one audio value.

In [ ]:
def mode_or_nan(s):
    m = s.mode()
    return m.iloc[0] if len(m) else np.nan

# owner per segment = the speaker that appears MOST in that segment's frames
owner = (df_all[df_all['has_audio']]
         .groupby('segment_id')['speaker']
         .agg(mode_or_nan))

df_all['seg_speaker']  = df_all['segment_id'].map(owner)
df_all['speaking_face'] = df_all['has_audio'] & (df_all['final_name'] == df_all['seg_speaker'])

dur = pd.DataFrame({'speaker': owner, 'duration': df_audio['duration']}).dropna()
print(dur.groupby('speaker')['duration'].describe()[['count', 'mean', '50%', 'max']])

In [ ]:
# plot the distribution of segment durations per speaker 
palette = ['#55A868', '#DD8452', '#4C72B0']

dur = pd.DataFrame({'speaker': owner, 'duration': df_audio['duration']}).dropna()
order = dur.groupby('speaker')['duration'].median().sort_values(ascending=False).index.tolist()

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=dur, x='speaker', y='duration', order=order, hue='speaker', 
            legend=False, palette=palette, width=0.5, showfliers=False, ax=ax)
sns.stripplot(data=dur, x='speaker', y='duration', order=order, 
              color='black', alpha=0.5, size=4, ax=ax)

for i, sp in enumerate(order):
    d = dur[dur['speaker'] == sp]['duration']
    ax.text(i, d.max() + 2, f'n={len(d)} segments \n median={d.median():.0f}s', ha='center', va='bottom', fontsize=9)

ax.set_title('Audio segment duration per speaker')
ax.set_xlabel('')
ax.set_ylabel('segment duration (s)')
ax.set_ylim(0, dur['duration'].max() * 1.18)
plt.tight_layout()
plt.show()

Segment-length asymmetry and its impact. The plot shows that Martins's audio segments are markedly longer than Gouveia Melo's. That means, Martins tends to give uninterrupted answers, while Gouveia Melo speaks in shorter bursts. This is a property of the debate, but it has two consequences for the segment-level analysis that follows:

- First, because we summarise each segment by averaging the speaker's emotion and pose over all its frames, longer segments are represented by flatter, more heavily-averaged values: within-segment emotional/pose variation is washed out more for Martins than for Gouveia Melo. The two candidates' segment-level observations are therefore not perfectly comparable —> Martins's points are coarser summaries of a longer span of behaviour.

- Second, the asymmetry produces an imbalanced number of observations (far fewer segments for Martins), which limits the statistical power of any per-candidate correlation involving her.

We keep the segment as the unit of analysis nonetheless, because the audio features are only defined per segment and splitting them would duplicate identical audio values. 

In [ ]:
def collapse(df, key='segment_id'):
    '''Collapse frame-level data to segment-level.'''
    agg = {c: 'mean' for c in emotion_cols + pose_feature_cols}   # face features: average over the segment
    agg.update({c: 'first' for c in NON_REDUNDANT})              # audio: same value all segment
    agg['seg_speaker']  = 'first'
    agg['final_name']   = 'first'
    agg['frame_number'] = 'count'
    out = (df.groupby(key).agg(agg)
             .rename(columns={'frame_number': 'n_frames'})
             .reset_index())
    out['dom_emotion'] = out[emotion_cols].idxmax(axis=1).str.replace('prob_', '', regex=False)
    return out

# SPEAKER view: one row per segment = the speaker's own face paired with their own audio
seg = collapse(df_all[df_all['speaking_face']])

print('segments:', len(seg))
print(seg['final_name'].value_counts())
seg.head(3)

## Auxiliar

> **▲ END — manual pipeline.** All cells between **▼ START** and here are replaced by the pipeline alternative above.

The variables `seg`, `emotion_cols`, and `pose_feature_cols` are now available regardless of which approach you used. The analysis sections below use only these three — `df_all` is only needed by Section A.

In [ ]:
emotion_colors = {'Anger':'tab:red','Happiness':'tab:olive','Neutral':'tab:gray',
                  'Sadness':'tab:blue','Surprise':'tab:orange','Fear':'tab:purple',
                  'Disgust':'tab:green','Contempt':'tab:brown'}
emotion_order  = ['Anger','Disgust','Contempt','Fear','Sadness','Surprise','Happiness','Neutral']


## A. Visual ↔ Visual  (frame level -> one row per face)

In [ ]:
# identified faces (drop 'uncertain'); reused by every frame-level visual analysis
ident = df_all[df_all['final_name'].notna() & (df_all['final_name'] != 'uncertain')].copy()

# people present in this debate
mods   = [p for p in ident['final_name'].unique() if p == 'moderator']
people = sorted(p for p in ident['final_name'].unique() if p != 'moderator') + mods
print('people in this debate:', people)

### A1. Emotion distribution per candidate

In [ ]:
fig, axes = plt.subplots(1, len(people), figsize=(5*len(people), 4), sharey=True)
axes = np.atleast_1d(axes)

for ax, p in zip(axes, people):

    person_rows = ident[ident['final_name'] == p] # all frames where person p appears
    emotions = person_rows['top_emotion'] 

    prop = emotions.value_counts(normalize=True) # proportion of frames per emotion (summing to 1)
    prop = prop.reindex(emotion_order, fill_value=0) # order by emotion_order, filling missing emotions with 0

    ax.bar(range(len(emotion_order)), prop.values,color=[emotion_colors[e] for e in emotion_order])
    ax.set_title(f'{p}  (n={len(person_rows)} frames)')
    ax.set_xticks(range(len(emotion_order)))
    ax.set_xticklabels(emotion_order, rotation=45, ha='right')

plt.suptitle('Emotion profiles per person (only frames where they appear)', fontsize=16)
plt.tight_layout()
plt.show()

### A2. Emotions over time

In [ ]:
palette = [ '#DD8452', '#4C72B0',  '#55A868']

person_colors = {p: palette[i] for i, p in enumerate(people)}

y_pos = {e: i for i, e in enumerate(emotion_order)}
# small vertical offset per person so they don't sit exactly on top of each other in a lane
offsets = dict(zip(people, [-0.15, 0, 0.15]))

fig, ax = plt.subplots(figsize=(15, 5))
for p in people:
    sub = ident[ident['final_name'] == p]
    y = sub['top_emotion'].map(y_pos) + offsets[p]
    ax.scatter(sub['frame_number'], y, s=8, alpha=0.5, color=person_colors[p], label=p)

ax.set_yticks(range(len(emotion_order)))
ax.set_yticklabels(emotion_order)
ax.set_xlabel('frame number (≈ seconds)')
ax.set_title('Top emotion over time, coloured by person')
ax.legend(title='person')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### A3. Poses and emotions

In [ ]:
# Doing for each cand to accout for camera angle differences, body propotions, etc. 

cands = [p for p in people if p != 'moderator']

sub = ident[ident['final_name'].isin(cands)] 

ordered_prob = [f'prob_{e}' for e in emotion_order]

for p in cands:
    sub = ident[ident['final_name'] == p]
    cc = sub[pose_feature_cols + ordered_prob].corr('spearman').loc[pose_feature_cols, ordered_prob]
    cc.columns = emotion_order
    plt.figure(figsize=(11, 3.5))
    sns.heatmap(cc, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=.5)
    plt.title(f'Pose vs Emotion — {p}  (n={len(sub)} frames)')
    plt.xlabel('emotion'); plt.ylabel('pose feature')
    plt.tight_layout(); plt.show()

For Gouveia, Fear and Surprise are almost nonexistent in his profile -> ignore those. His trustworthy cells are on Disgust/Sadness (which he does show): body_openness ↔ Disgust 0.28, shoulder_slope ↔ Disgust 0.32, head_tilt ↔ Sadness 0.28. Reading those, he opens up and squares his shoulders during disgust and tilts his head during sadness.

For Martins, Anger and Disgust are rare, so wrist_height ↔ Anger/Disgust (0.22, 0.26) is unreliable. Her common emotions are Sadness and Surprise, where wrist_height ↑ Surprise (0.22) / ↓ Sadness (−0.18) and body_openness ↑ Surprise (0.15) are the real (but modest) signals.

conclusion: within-person pose↔emotion coupling is weak and person-specific, strongest on body_openness, and Gouveia shows a bit more structure than Martins. 

### A4. REVER ISTO Emotion co-occurrence between the two candidates in shared frames

In [ ]:
cands = [p for p in people if p != 'moderator']
g, m = cands[0], cands[1]

sub = ident[ident['final_name'].isin(cands)] # only frames where we see either candidate

# one emotion per (frame, person)
per_frame_person = sub.groupby(['frame_number', 'final_name'])['top_emotion'].first()
# pivot persons into columns -> one row per frame, one column per candidate
piv = per_frame_person.unstack('final_name')

both = piv.dropna(subset=cands)
print('frames with BOTH candidates on screen:', len(both))

keep = [e for e in emotion_order
        if (both[g] == e).sum() >= 10 or (both[m] == e).sum() >= 10]
ct = pd.crosstab(both[g], both[m]).reindex(index=keep, columns=keep, fill_value=0)

total = ct.values.sum() # total frames with both candidates and a valid emotion

expected = ct.sum(1).values.reshape(-1, 1) @ ct.sum(0).values.reshape(1, -1) / total # expected counts under independence = (row total * column total) / grand total

lift = (ct / expected.clip(min=1e-9)).mask(expected < 5) # observed / expected; clip expected to avoid division by zero (if expected is 0, then observed must also be 0, so this cell will show 0 lift)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Co-occurrence counts')
sns.heatmap(lift, annot=True, fmt='.2f', cmap='coolwarm', center=1, vmin=0, vmax=2, ax=axes[1])
axes[1].set_title('Observed / Expected  (diagonal = same emotion)')

for ax in axes:
    ax.set_xlabel(m)
    ax.set_ylabel(g)
plt.tight_layout() 
plt.show()

from scipy.stats import chi2_contingency
chi2, p, dof, _ = chi2_contingency(ct)
v = np.sqrt(chi2 / (total * (min(ct.shape) - 1)))
print(f'chi2={chi2:.1f}  p={p:.3g}  Cramér V={v:.3f}')

We restrict to frames where both candidates are on screen and cross-tabulate their top emotions. 

The left panel shows the raw co-occurrence counts; 

The right panel shows observed/expected ("lift"), where a value above 1 means a pair occurs more often than it would if the two candidates' expressions were independent (cells with too few expected occurrences are left blank). 

The association is statistically significant and moderate (Cramér's V = 0.325, p ≪ 0.001), so their on-screen expressions are clearly not independent. 

However, the effect is dominated by the Neutral↔Neutral cell (lift ≈ 7), which alone accounts for roughly two-thirds of the chi-square: both candidates tend to be calm at the same time and animated at the same time. Much of this likely reflects the shared rhythm of the debate and the director's editing (both shown together during the same moments) rather than a direct emotional interaction.

In [ ]:
expr = [e for e in ct.index if e != 'Neutral']
ct2 = ct.loc[expr, expr]
total2 = ct2.values.sum()
exp2 = ct2.sum(1).values.reshape(-1, 1) @ ct2.sum(0).values.reshape(1, -1) / total2
lift2 = (ct2 / exp2.clip(min=1e-9)).mask(exp2 < 5)

plt.figure(figsize=(6, 5))
sns.heatmap(lift2, annot=True, fmt='.2f', cmap='coolwarm', center=1, vmin=0, vmax=2)
plt.title('Co-occurrence lift, Neutral excluded'); plt.xlabel(m); plt.ylabel(g)
plt.tight_layout(); plt.show()

from scipy.stats import chi2_contingency
chi2b, pb, _, _ = chi2_contingency(ct2)
print(f'expressive-only: chi2={chi2b:.1f}  p={pb:.3g}  '
      f'Cramér V={np.sqrt(chi2b/(total2*(min(ct2.shape)-1))):.3f}')

To isolate genuine emotion-to-emotion coupling among engaged moments, we repeat the analysis with Neutral removed from both axes. 

The association drops sharply, from V = 0.325 to V = 0.137 — a weak effect (still significant only because of the large number of frames; here the effect size matters more than the p-value). 

This confirms that most of the original association was the shared calm/engaged tempo rather than specific pairings of expressions. 

The structure that remains is a mild tendency for negative, disdainful expressions to align: Gouveia Melo's Contempt and Disgust co-occur with Martins's Contempt (lift ≈ 1.6–1.7) and with her Happiness (≈ 1.3–1.5), consistent with one candidate smiling or smirking while the other shows disdain.

There is also an asymmetric Surprise (Gouveia) ↔ Sadness (Martins) pairing (≈ 1.6) and essentially no shared surprise, while Gouveia Melo's Sadness is unrelated to Martins's expression. Given the noise in the emotion labels and the fact that this is co-occurrence rather than proven influence, these residual effects should be read as suggestive.

## B. Audio ↔ Visual  (segment level —> one row per segment)

### B1. Audio ↔ Emotion

In [ ]:
cands = [p for p in people if p != 'moderator']
segc = seg[seg['final_name'].isin(cands)].copy()
ordered_prob = [f'prob_{e}' for e in emotion_order]

# POOLED (raw): mixes between-candidate baselines with within-candidate change
pooled = segc[NON_REDUNDANT + ordered_prob].corr('spearman').loc[NON_REDUNDANT, ordered_prob]

# WITHIN candidate: remove each candidate's own mean first (kills voice/sex baseline)
cen = segc.copy()
for col in NON_REDUNDANT + ordered_prob:
    cen[col] = segc[col] - segc.groupby('final_name')[col].transform('mean')
within = cen[NON_REDUNDANT + ordered_prob].corr('spearman').loc[NON_REDUNDANT, ordered_prob]

for mat, ttl in [(pooled, 'POOLED (raw)'), (within, 'WITHIN candidate (baseline removed)')]:
    mat.columns = emotion_order
    plt.figure(figsize=(12, 4.5))
    sns.heatmap(mat, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=.5)
    plt.title(f'Audio vs Emotion — {ttl}  (candidates only, n={len(segc)})')
    plt.xlabel('emotion'); plt.ylabel('audio feature')
    plt.tight_layout(); plt.show()

We correlate the segment-level audio features with the speaker's averaged facial-emotion probabilities, comparing a naive pooled correlation against a within-candidate version (each candidate's own mean subtracted from every feature first, removing baseline voice and resting-expression differences).

The pooled heatmap shows strong correlations (|r| up to 0.73), but they are almost entirely an artifact of identity: the emotions split into two blocks with opposite audio signatures — Anger/Disgust/Contempt versus Fear/Surprise/Happiness — which map directly onto the two candidates' emotion profiles (the first is characteristic of Gouveia Melo, the second of Martins). In other words, the correlation is separating the two speakers, who differ in both their typical facial expression and their baseline voice, rather than capturing emotion-driven changes in speech. The clearest evidence is that Sadness and Neutral — the emotions both candidates share — sit near zero, because they cannot distinguish the two people. 

Once each candidate's baseline is removed, the correlations collapse to |r| < 0.3 for almost all cells. The only genuine within-person signal is a weak  trace: pitch (meanF0Hz) rises slightly with Fear, Surprise and Anger relative to a speaker's own norm, and shimmer increases with Disgust (≈ 0.39, the only cell that would survive a multiple-comparison correction given the small number of segments). 

Conclsuion: within a speaker, the audio features carry only a weak reflection of facial emotion.
These within-person effects are limited by the small number of on-camera segments available in a single debate (~80), and noise in the facial-emotion labels attenuates correlations toward zero, so the weak coupling observed here may understate the true relationship. The multi-video analysis — pooling segments across all debates, with the centering kept within each speaker (and debate) to preserve the same control — provides a more powerful test of whether this arousal-related audio–emotion link is consistent and reliably detectable, even if the effect size itself is likely to remain modest.

### B5. Audio ↔ Pose (speaking face only)

# Is one modality enough?

In [ ]:
from sklearn.cross_decomposition import CCA
from scipy.stats import pearsonr

visual_features = pose_feature_cols + emotion_cols
audio_features  = NON_REDUNDANT
d = seg[visual_features + audio_features].dropna()
print('segments used:', len(d))

# structure of each modality space
mods = {'Visual (pose+emotion)': visual_features, 'Audio': audio_features,
        'All combined': visual_features + audio_features}
fig, axes = plt.subplots(1, 3, figsize=(21,6))
for ax, (nm, fs) in zip(axes, mods.items()):
    X2 = PCA(2, random_state=42).fit_transform(StandardScaler().fit_transform(d[fs]))
    ax.scatter(X2[:,0], X2[:,1], c='steelblue', alpha=0.4, s=15, edgecolors='none')
    ax.set_title(nm); ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.grid(alpha=0.5)
plt.suptitle('PCA structure per modality (segments)'); plt.tight_layout(); plt.show()

# how much do the modalities actually SHARE?
Xv = StandardScaler().fit_transform(d[visual_features])
Xa = StandardScaler().fit_transform(d[audio_features])
ncomp = min(3, Xv.shape[1], Xa.shape[1])
Uv, Ua = CCA(n_components=ncomp).fit(Xv, Xa).transform(Xv, Xa)
print('\nCanonical correlations (visual vs audio):')
for i in range(ncomp):
    r, p = pearsonr(Uv[:,i], Ua[:,i])
    print(f'  component {i+1}: r = {r:.3f}  (p = {p:.4f})')
print('\nLow values -> modalities carry different information -> one modality is NOT enough.')